# Multinomial Logistic Regression for Bigram Language Modeling

## Introduction
This notebook implements a **Multinomial Logistic Regression** model for bigram language modeling. The goal is to estimate the probability $P(\text{word}_2 \mid \text{word}_1)$, where $\text{word}_1$ is the previous word and $\text{word}_2$ is the next word. The model uses a single-layer neural network with a softmax activation function.

## Theory
### Problem Setup
The goal is to estimate the probability $P(\text{word}_2 \mid \text{word}_1)$, where $\text{word}_1$ is the previous word and $\text{word}_2$ is the next word. The input is a one-hot encoded vector representing $\text{word}_1$, and the output is a probability distribution over the vocabulary for $\text{word}_2$.

### Notation
- Let the vocabulary size be $V$.
- Let $\mathbf{x} \in \mathbb{R}^V$ be the one-hot encoded input vector for $\text{word}_1$. For example, if $\text{word}_1$ is the $i$-th word in the vocabulary, then:
  $$
  \mathbf{x} = [0, 0, \dots, 1, \dots, 0]^\top,
  $$
  where the $i$-th element is 1 and all others are 0.
- Let $\mathbf{W} \in \mathbb{R}^{V \times V}$ be the weight matrix, where each row corresponds to the weights for predicting a specific word in the vocabulary.
- Let $\mathbf{b} \in \mathbb{R}^V$ be the bias vector.
- Let $\mathbf{z} \in \mathbb{R}^V$ be the logits (unnormalized scores) for each word in the vocabulary.

### Forward Pass
- Compute the logits $\mathbf{z}$:
  $$
  \mathbf{z} = \mathbf{W}^\top \mathbf{x} + \mathbf{b}.
  $$
  Since $\mathbf{x}$ is one-hot encoded, this simplifies to selecting the $i$-th row of $\mathbf{W}$ (where $\mathbf{x}_i = 1$):
  $$
  \mathbf{z} = \mathbf{W}_i + \mathbf{b},
  $$
  where $\mathbf{W}_i$ is the $i$-th row of $\mathbf{W}$.
- Apply the softmax function to convert logits $\mathbf{z}$ into probabilities:
  $$
  P(\text{word}_2 = j \mid \text{word}_1 = i) = \text{softmax}(\mathbf{z})_j = \frac{e^{z_j}}{\sum_{k=1}^V e^{z_k}}.
  $$

### Training
The model is trained using **cross-entropy loss**. For a single training example $(\text{word}_1, \text{word}_2)$, the loss is:
$$
\mathcal{L} = -\log P(\text{word}_2 \mid \text{word}_1).
$$
The gradients of the loss with respect to the parameters ($\mathbf{W}$ and $\mathbf{b}$) are computed using backpropagation, and the parameters are updated using gradient descent.

### Gradient Computation
- The gradient of the loss with respect to the logits $\mathbf{z}$ is:
  $$
  \frac{\partial \mathcal{L}}{\partial z_j} = P(\text{word}_2 = j \mid \text{word}_1) - \mathbb{I}(\text{word}_2 = j),
  $$
  where $\mathbb{I}(\text{word}_2 = j)$ is an indicator function that is 1 if $\text{word}_2$ is the $j$-th word and 0 otherwise.
- The gradients with respect to $\mathbf{W}$ and $\mathbf{b}$ are:
  $$
  \frac{\partial \mathcal{L}}{\partial \mathbf{W}_j} = \left( P(\text{word}_2 = j \mid \text{word}_1) - \mathbb{I}(\text{word}_2 = j) \right) \cdot \mathbf{x},
  $$
  $$
  \frac{\partial \mathcal{L}}{\partial b_j} = P(\text{word}_2 = j \mid \text{word}_1) - \mathbb{I}(\text{word}_2 = j).
  $$

## Step 1: Prepare the Data
We will use a larger corpus of sentences to train the model. Each sentence is tokenized into words, and bigrams are extracted.

In [ ]:
import numpy as np

# Define a larger corpus of sentences (excerpt from Shakespeare's works)
corpus = [
    "To be or not to be that is the question",
    "Whether tis nobler in the mind to suffer",
    "The slings and arrows of outrageous fortune",
    "Or to take arms against a sea of troubles",
    "And by opposing end them",
    "To die to sleep no more",
    "And by a sleep to say we end",
    "The heartache and the thousand natural shocks",
    "That flesh is heir to tis a consummation",
    "Devoutly to be wished"
]

# Tokenize the sentences into words
tokenized_corpus = [sentence.split() for sentence in corpus]

# Create a vocabulary
vocab = set()
for sentence in tokenized_corpus:
    vocab.update(sentence)
vocab = list(vocab)  # Convert to list for indexing
vocab_size = len(vocab)

# Create word-to-index and index-to-word mappings
word_to_idx = {word: idx for idx, word in enumerate(vocab)}
idx_to_word = {idx: word for word, idx in word_to_idx.items()}

# Extract bigrams from the corpus
bigrams = []
for sentence in tokenized_corpus:
    for i in range(len(sentence) - 1):
        bigrams.append((sentence[i], sentence[i + 1]))

print("Vocabulary:", vocab)
print("Number of bigrams:", len(bigrams))

## Step 2: One-Hot Encoding
Each word is represented as a one-hot encoded vector.

In [ ]:
def one_hot_encode(word, vocab_size, word_to_idx):
    vec = np.zeros(vocab_size)
    vec[word_to_idx[word]] = 1
    return vec

# Example: One-hot encode the word "to"
one_hot_encode("to", vocab_size, word_to_idx)

## Step 3: Initialize Model Parameters
The model consists of a weight matrix $\mathbf{W}$ and a bias vector $\mathbf{b}$.

In [ ]:
# Initialize weight matrix and bias vector
np.random.seed(42)
W = np.random.randn(vocab_size, vocab_size) * 0.01  # Weight matrix
b = np.zeros(vocab_size)  # Bias vector

## Step 4: Forward Pass
Compute the logits $\mathbf{z}$ and apply the softmax function to obtain probabilities.

In [ ]:
def softmax(z):
    exp_z = np.exp(z - np.max(z))  # Numerical stability
    return exp_z / np.sum(exp_z)

def forward_pass(x, W, b):
    z = np.dot(W.T, x) + b  # Logits
    p = softmax(z)  # Probabilities
    return z, p

# Example: Forward pass for the bigram ("to", "be")
x = one_hot_encode("to", vocab_size, word_to_idx)
z, p = forward_pass(x, W, b)
print("Logits:", z)
print("Probabilities:", p)

## Step 5: Training the Model
The model is trained using **cross-entropy loss** and gradient descent.

In [ ]:
def cross_entropy_loss(p, target_idx):
    return -np.log(p[target_idx])

def compute_gradients(x, p, target_idx):
    # Gradient of the loss with respect to the logits
    dL_dz = p.copy()
    dL_dz[target_idx] -= 1

    # Gradient of the loss with respect to W and b
    dL_dW = np.outer(x, dL_dz)
    dL_db = dL_dz

    return dL_dW, dL_db

# Training loop
learning_rate = 0.1
epochs = 100

for epoch in range(epochs):
    total_loss = 0
    for word1, word2 in bigrams:
        # One-hot encode input and target
        x = one_hot_encode(word1, vocab_size, word_to_idx)
        target_idx = word_to_idx[word2]

        # Forward pass
        z, p = forward_pass(x, W, b)

        # Compute loss
        loss = cross_entropy_loss(p, target_idx)
        total_loss += loss

        # Compute gradients
        dL_dW, dL_db = compute_gradients(x, p, target_idx)

        # Update parameters
        W -= learning_rate * dL_dW
        b -= learning_rate * dL_db

    # Print average loss for the epoch
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}, Loss: {total_loss / len(bigrams)}")

## Step 6: Generate Predictions
After training, the model can predict the next word given a previous word.

In [ ]:
def predict_next_word(word, W, b, vocab_size, word_to_idx, idx_to_word):
    x = one_hot_encode(word, vocab_size, word_to_idx)
    z, p = forward_pass(x, W, b)
    next_word_idx = np.argmax(p)
    return idx_to_word[next_word_idx]

# Example: Predict the next word after "to"
print("Next word after 'to':", predict_next_word("to", W, b, vocab_size, word_to_idx, idx_to_word))

## Step 7: Visualize the Learned Probabilities
We can visualize the learned probabilities for a given input word.

In [ ]:
import matplotlib.pyplot as plt

def plot_probabilities(word, W, b, vocab_size, word_to_idx, idx_to_word):
    x = one_hot_encode(word, vocab_size, word_to_idx)
    z, p = forward_pass(x, W, b)
    plt.bar(range(vocab_size), p)
    plt.xticks(range(vocab_size), [idx_to_word[i] for i in range(vocab_size)], rotation=90)
    plt.xlabel("Words")
    plt.ylabel("Probability")
    plt.title(f"Next Word Probabilities after '{word}'")
    plt.show()

# Example: Visualize probabilities after "to"
plot_probabilities("to", W, b, vocab_size, word_to_idx, idx_to_word)

## Conclusion
This notebook demonstrates how to implement a **Multinomial Logistic Regression** model for bigram language modeling. The model is trained on a larger corpus and can predict the next word given a previous word. The learned probabilities can be visualized to understand the model's behavior.